In [20]:

#!/usr/bin/env python3
import numpy as np
import pandas as pd
from scipy.stats import gamma
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score
import importlib

# List of PyOD models you want to compare.
# Make sure you have pyod installed: pip install pyod
model_specs = [
    ('KNN',   'knn',   {'n_neighbors': 5,'p':1}),
    ('LOF',   'lof',   {'n_neighbors': 5, 'novelty': True}),
    ('SLOF',  'slof',  {'n_neighbors': 5, 'contamination': 0.1}),
    ('LoOP',  'loop',  {'n_neighbors': 5, 'contamination': 0.1}),
    ('LDOF',  'ldof',  {'n_neighbors': 5, 'contamination': 0.1}),
    ('ABOD',  'abod',  {'contamination': 0.1}),
    ('LDF',   'ldf',   {'n_neighbors': 5, 'contamination': 0.1}),
    ('INFLO', 'inflo', {'n_neighbors': 5, 'contamination': 0.1}),
    ('COF',   'cof',   {'n_neighbors': 5, 'contamination': 0.1}),
]

# Build a dict mapping method name → builder(train_data)→model
detectors = {}
for name, module_name, params in model_specs:
    try:
        module = importlib.import_module(f'pyod.models.{module_name}')
        cls = getattr(module, name)
        detectors[name] = (lambda C, P: (lambda tr: C(**P).fit(tr)))(cls, params)
    except (ImportError, AttributeError):
        print(f"Skipping {name}: pyod.models.{module_name} not found")

# Always include the parametric CDF rule
detectors['CDF'] = None

# Simulation parameters
runs     = 500
n_train  = 200
n_test   = 200
a_in     = 2.0
scale_in = 2.0
a_out    = 5.0
scale_out= 2.0

# Storage for AUCs
aucs = {name: np.zeros(runs) for name in detectors}

for i in range(runs):
    # 1) Generate training inliers and fit Gamma MLE
    train_in = np.random.gamma(a_in, scale_in, size=n_train)
    shape_hat, _, scale_hat = gamma.fit(train_in, floc=0)
    
    # 2) Generate test inliers + outliers
    test_in  = np.random.gamma(a_in, scale_in, size=n_test)
    test_out = np.random.gamma(a_out, scale_out, size=n_test)
    X_test = np.concatenate([test_in, test_out]).reshape(-1,1)
    y_true = np.concatenate([np.zeros(n_test), np.ones(n_test)])
    
    # 3) Evaluate each method
    for name, builder in detectors.items():
        if name == 'CDF':
            # Parametric CDF score
            scores = np.concatenate([
                gamma.cdf(test_in,  a=shape_hat, loc=0, scale=scale_hat),
                gamma.cdf(test_out, a=shape_hat, loc=0, scale=scale_hat)
            ])
        else:
            model = builder(train_in.reshape(-1,1))
            scores = model.decision_function(X_test)
        aucs[name][i] = roc_auc_score(y_true, scores)

# Summarize results
df = pd.DataFrame(aucs)
summary = df.describe().T[['mean','std','min','25%','50%','75%','max']]

print("\nMonte Carlo AUC Comparison (500 runs)\n")
print(summary.to_string())


Skipping SLOF: pyod.models.slof not found
Skipping LoOP: pyod.models.loop not found
Skipping LDOF: pyod.models.ldof not found
Skipping LDF: pyod.models.ldf not found
Skipping INFLO: pyod.models.inflo not found

Monte Carlo AUC Comparison (500 runs)

          mean       std       min       25%       50%       75%       max
KNN   0.847339  0.022329  0.774825  0.832556  0.846800  0.862381  0.913325
LOF   0.623310  0.059925  0.441175  0.581287  0.622781  0.668391  0.754725
ABOD  0.805949  0.024521  0.725750  0.790969  0.806975  0.822325  0.869150
COF   0.502065  0.026200  0.425000  0.483131  0.503575  0.519853  0.581300
CDF   0.890580  0.015399  0.819125  0.880663  0.892000  0.899925  0.935075


In [8]:
import sys
print(sys.version)

3.11.7 (tags/v3.11.7:fa7a6f2, Dec  4 2023, 19:24:49) [MSC v.1937 64 bit (AMD64)]


In [16]:
summary.to_csv("Monte Carlo Simulation.csv")